In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve)

In [3]:
df = pd.read_csv(r"C:\Users\shobh\Downloads\AI and DATA SCIENCE\𝗗𝗲𝗲𝗽 𝗟𝗲𝗮𝗿𝗻𝗶𝗻𝗴 𝗣𝗿𝗼𝗷𝗲𝗰𝘁𝘀\Smartphone-Addiction-Prediction\data\Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv")
print(df.shape)
df.head()

(7500, 16)


,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,NaN,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,NaN,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1


In [ ]:
df.info()

In [9]:
df_clean = df.drop(columns=["transaction_id", "user_id", "addiction_level"])

df_clean = df_clean.drop_duplicates()
print("Duplicates found:", df.duplicated().sum())
print("Cleaned shape:", df_clean.shape)

Duplicates found: 0
Cleaned shape: (7500, 13)


In [8]:
df_clean = df

df_clean

,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
0,TXN00001,U00001,21,Male,3.23,2.01,0.89,4.55,7.55,248,154,3.95,Medium,Yes,NaN,0
1,TXN00002,U00002,24,Other,5.09,3.81,2.24,4.44,7.66,127,71,6.71,Medium,Yes,NaN,0
2,TXN00003,U00003,31,Other,6.06,1.36,3.83,2.35,4.92,44,106,8.68,High,No,Mild,0
3,TXN00004,U00004,32,Other,7.83,5.85,1.51,3.54,8.23,178,107,9.77,High,Yes,Moderate,1
4,TXN00005,U00005,25,Male,9.96,5.92,3.42,5.27,6.21,136,177,12.55,Low,No,Severe,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7495,TXN07496,U07496,26,Other,9.85,1.75,3.13,3.49,5.81,249,122,11.99,Low,Yes,Moderate,1
7496,TXN07497,U07497,35,Male,5.67,2.33,2.76,5.90,8.47,197,56,7.08,Low,No,NaN,0
7497,TXN07498,U07498,22,Female,9.99,3.61,1.09,1.16,8.17,207,24,12.84,Medium,Yes,Severe,1
7498,TXN07499,U07499,23,Male,8.74,1.59,0.07,4.64,6.19,134,62,10.52,High,Yes,Severe,1


In [7]:
df.sample()

,transaction_id,user_id,age,gender,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,stress_level,academic_work_impact,addiction_level,addicted_label
6262,TXN06263,U06263,25,Female,4.06,5.93,2.07,4.68,6.89,65,130,6.08,Medium,No,Severe,1


In [15]:
categorical_cols = ["gender", "stress_level", "academic_work_impact"]

df_encoded = df.copy()
encoders = {}
for col in categorical_cols:
    dummies = pd.get_dummies(df_encoded[col], prefix=col, dtype="int64")
    encoders[col] = list(dummies.columns)
    df_encoded = pd.concat([df_encoded.drop(columns=[col]), dummies], axis=1)

df_encoded.head()

,transaction_id,user_id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,...,addiction_level,addicted_label,gender_Female,gender_Male,gender_Other,stress_level_High,stress_level_Low,stress_level_Medium,academic_work_impact_No,academic_work_impact_Yes
0,TXN00001,U00001,21,3.23,2.01,0.89,4.55,7.55,248,154,...,NaN,0,0,1,0,0,0,1,0,1
1,TXN00002,U00002,24,5.09,3.81,2.24,4.44,7.66,127,71,...,NaN,0,0,0,1,0,0,1,0,1
2,TXN00003,U00003,31,6.06,1.36,3.83,2.35,4.92,44,106,...,Mild,0,0,0,1,1,0,0,1,0
3,TXN00004,U00004,32,7.83,5.85,1.51,3.54,8.23,178,107,...,Moderate,1,0,0,1,1,0,0,0,1
4,TXN00005,U00005,25,9.96,5.92,3.42,5.27,6.21,136,177,...,Severe,1,0,1,0,0,1,0,1,0


In [ ]:
Load Data
      │
      ▼
Preprocessing
      │
      ▼
Train-Test Split
      │
      ▼
Tensor Conversion
      │
      ▼
TensorDataset
      │
      ▼
DataLoader
      │
      ▼
Design ANN
      │
      ▼
Loss Function
      │
      ▼
Optimizer
      │
      ▼
Training Loop
      │
      ▼
Evaluation
      │
      ▼
Save Model

In [36]:
#building first dl neural network 


#X and y 


X = df_encoded.drop(columns = ['transaction_id','user_id','addiction_level','addicted_label'])

y = df_encoded['addicted_label']


#test-test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y     
)



#now convert everything to tensor 


X_train = torch.FloatTensor(X_train.values)
X_test = torch.FloatTensor(X_test.values)

y_train = torch.LongTensor(y_train.values)
y_test = torch.LongTensor(y_test.values)


#now lets create dataset to create batches 


from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


#after creating datasets lets create batches using dataloader


train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)

test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)


#after data preprocessing lets design the neural network architecture

print(X_train.shape) # 17 input features in our dataset 


import torch.nn as nn

model = nn.Sequential(
    nn.Linear(17, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),

    nn.Linear(8, 2)

)

#now loss function 


criterion = nn.CrossEntropyLoss()



#now optimiser (using adam for now )


optimizer = torch.optim.Adam(model.parameters(),lr=0.001)


#now training loop 


epochs = 30

for epoch in range(epochs):

    # Set model to training mode
    model.train()

    # Store total loss of the epoch
    running_loss = 0.0

    # Loop through batches
    for X_batch, y_batch in train_loader:

        # Forward Pass
        outputs = model(X_batch)

        # Calculate Loss
        loss = criterion(outputs, y_batch)

        # Clear Previous Gradients
        optimizer.zero_grad()

        # Backward Pass
        loss.backward()

        # Update Weights
        optimizer.step()

        # Add batch loss
        running_loss += loss.item()

    # Print Average Loss of the Epoch
    avg_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {avg_loss:.4f}")

torch.Size([6000, 17])
Epoch [1/30]  Loss: 0.8095
Epoch [2/30]  Loss: 0.4958
Epoch [3/30]  Loss: 0.3927
Epoch [4/30]  Loss: 0.3531
Epoch [5/30]  Loss: 0.3205
Epoch [6/30]  Loss: 0.3084
Epoch [7/30]  Loss: 0.3042
Epoch [8/30]  Loss: 0.3025
Epoch [9/30]  Loss: 0.2986
Epoch [10/30]  Loss: 0.2898
Epoch [11/30]  Loss: 0.2931
Epoch [12/30]  Loss: 0.2947
Epoch [13/30]  Loss: 0.2879
Epoch [14/30]  Loss: 0.2853
Epoch [15/30]  Loss: 0.2846
Epoch [16/30]  Loss: 0.2801
Epoch [17/30]  Loss: 0.2796
Epoch [18/30]  Loss: 0.2795
Epoch [19/30]  Loss: 0.2790
Epoch [20/30]  Loss: 0.2820
Epoch [21/30]  Loss: 0.2760
Epoch [22/30]  Loss: 0.2722
Epoch [23/30]  Loss: 0.2727
Epoch [24/30]  Loss: 0.2681
Epoch [25/30]  Loss: 0.2706
Epoch [26/30]  Loss: 0.2741
Epoch [27/30]  Loss: 0.2744
Epoch [28/30]  Loss: 0.2692
Epoch [29/30]  Loss: 0.2688
Epoch [30/30]  Loss: 0.2698


In [40]:
#now model evaluation 


model.eval()

with torch.no_grad():

    outputs = model(X_test)

    predictions = torch.argmax(outputs, dim=1)


print(predictions)

tensor([0, 1, 0,  ..., 1, 1, 1])
